# THUISBATTERIJ PLUS DYNAMISCH ENERGIECONTRACT EN POSTCODEROOS DEELNAME

Onderzoeksvraag:
Wat is de terugverdientijd van een thuisbatterij bij verschillende contracten.
- vast contract
- dynamisch contract
- dynamisch contract + postcoderoos voordeel

In [397]:
# INLEZEN DATA BESTANDEN EN CONSTANTES

from pathlib import Path
import pandas as pd

VERBRUIK_CSV = (Path.cwd() / "../energieverbruik/energie_verbruik_2025Q3_2026Q2.csv").resolve()
PRIJZEN_CSV = (Path.cwd() / "../energieprijzen/dynamische_stroomprijzen_jeroenpuntnl_GreenChoice_2025-07-01_2026-07-01_all_in.csv").resolve()

BATTERIJ_CAPACITEIT = 2 # kWh
LAAD_VERMOGEN = 2.5 # kW
RTE = 0.9  # round-trip efficiency
BATTERIJ_PRIJS_PER_KWH = 500  # euro per kWh
BATTERIJ_PRIJS = 250 + BATTERIJ_PRIJS_PER_KWH * BATTERIJ_CAPACITEIT

def read_csv_to_dataframe(csv_path: Path) -> pd.DataFrame:
    print(f"Reading file into a pandas table: {csv_path}")
    return pd.read_csv(csv_path, encoding="utf-8")

try:
    verbruikTable = read_csv_to_dataframe(VERBRUIK_CSV)
    print(f"Rows in verbruikTable: {len(verbruikTable)}, elements per row: {len(verbruikTable.columns)}")
    # print(verbruikTable.head())

    print()
    
    prijzenTable = read_csv_to_dataframe(PRIJZEN_CSV)
    print(f"Rows in prijzenTable: {len(prijzenTable)}, elements per row: {len(prijzenTable.columns)}")
    # print(prijzenTable.tail())
except Exception as exc:
    print(f"Error loading pandas table: {exc}")

Reading file into a pandas table: /Users/felixdonkers/Library/Mobile Documents/com~apple~CloudDocs/my_python_projects/misc_projects/thuisbatterij/energieverbruik/energie_verbruik_2025Q3_2026Q2.csv
Rows in verbruikTable: 365, elements per row: 101

Reading file into a pandas table: /Users/felixdonkers/Library/Mobile Documents/com~apple~CloudDocs/my_python_projects/misc_projects/thuisbatterij/energieprijzen/dynamische_stroomprijzen_jeroenpuntnl_GreenChoice_2025-07-01_2026-07-01_all_in.csv
Rows in prijzenTable: 365, elements per row: 101


## Berekenen energiekosten met vast contract

In [398]:
from pathlib import Path
import pandas as pd

excluded_columns = {"date", "total", "minimum", "average", "maximum"}

def sum_dataframe_values(table: pd.DataFrame) -> float:
    data_columns = [col for col in table.columns if col.lower() not in excluded_columns]
    numeric_table = table[data_columns].apply(pd.to_numeric, errors="coerce")
    return float(numeric_table.sum().sum())

energieUsage = sum_dataframe_values(verbruikTable)
energy_price = 0.27  # €/kWh
totalCost = energieUsage * energy_price

print("Energie prijs:", energy_price, "[€/kWh]")
print("Energie gebruik:", energieUsage, "[kWh]")
print("Energiekosten met vast contract: €", f"{totalCost:.2f}")

Energie prijs: 0.27 [€/kWh]
Energie gebruik: 4591.392 [kWh]
Energiekosten met vast contract: € 1239.68


# Bereken energiekosten met dynamisch contract

In [399]:
def multiply_tables_by_column(verbruik_table: pd.DataFrame, prijzen_table: pd.DataFrame) -> pd.DataFrame:
    shared_columns = [
        col for col in verbruik_table.columns
        if col.lower() not in excluded_columns and col in prijzen_table.columns
    ]
    verbruik_numeric = verbruik_table[shared_columns].apply(pd.to_numeric, errors="coerce")
    prijzen_numeric = prijzen_table[shared_columns].apply(pd.to_numeric, errors="coerce")
    return verbruik_numeric.mul(prijzen_numeric)

resultTable = multiply_tables_by_column(verbruikTable, prijzenTable)
total_dyn_costs = sum_dataframe_values(resultTable)

print("Energiekosten met dynamisch contract: €", f"{total_dyn_costs:.2f}, voordeel = €", f"{totalCost - total_dyn_costs:.2f}")

Energiekosten met dynamisch contract: € 1142.14, voordeel = € 97.54


## Berekenen energiekosten met dynamisch contract EN batterij


In [400]:
#code
def create_charge_table(prijzen_table: pd.DataFrame) -> pd.DataFrame:
    data_columns = [col for col in prijzen_table.columns if col.lower() not in excluded_columns]
    charge_table_rows = []

    for row_index, row in prijzen_table[data_columns].iterrows():
        numeric_row = pd.to_numeric(row, errors="coerce")
        charge_columns = numeric_row.sort_values().index.tolist()
        charge_table_rows.append({
            "row_index": row_index,
            "sorted_price_columns": charge_columns,
        })
        # print(f"Row {row_index}: sorted columns = {sorted_columns}")
    return pd.DataFrame(charge_table_rows)

def create_discharge_table(prijzen_table: pd.DataFrame) -> pd.DataFrame:
    data_columns = [col for col in prijzen_table.columns if col.lower() not in excluded_columns]
    discharge_table_rows = []

    for row_index, row in prijzen_table[data_columns].iterrows():
        numeric_row = pd.to_numeric(row, errors="coerce")
        discharge_columns = numeric_row.sort_values(ascending=False).index.tolist()
        discharge_table_rows.append({
            "row_index": row_index,
            "sorted_price_columns": discharge_columns,
        })
        # print(f"Row {row_index}: sorted columns = {sorted_columns}")
    return pd.DataFrame(discharge_table_rows)

def process_charging_tables(
    verbruikTable: pd.DataFrame, 
    charge_timestamps: pd.DataFrame,
    discharge_timestamps: pd.DataFrame,
    charge_speed: float,
    battery_capacity_max: float,
    rte: float
) -> pd.DataFrame:
    energy_per_quarter = charge_speed / 4
    nrof_charge_intervals = battery_capacity_max / energy_per_quarter
    print(f"Battery capacity: {battery_capacity_max} kWh, charge speed: {charge_speed} kW")
    print(f"Number of charge intervals [15 min.]: {nrof_charge_intervals}, energy per interval: {energy_per_quarter} kWh")
    verbruikTableCopy = verbruikTable.copy()

    for row_index, row in verbruikTableCopy.iterrows():
        # process charging    
        available_columns = charge_timestamps.loc[row_index, "sorted_price_columns"]
        battery_capacity = 0
        count = 0
        for col_name in available_columns:
            if count >= int(nrof_charge_intervals):
                charge = energy_per_quarter*(nrof_charge_intervals - count)
                verbruikTableCopy.loc[row_index, col_name] += charge
                battery_capacity += charge
                # print(f"count={count}, battery capacity now: {battery_capacity} kWh")
                break
            else:
                verbruikTableCopy.loc[row_index, col_name] += energy_per_quarter
                battery_capacity += energy_per_quarter
                count += 1

        # process discharging
        battery_capacity_remaining = battery_capacity * rte;
        discharge_columns = discharge_timestamps.loc[row_index, "sorted_price_columns"]
        for col_name in discharge_columns:
            verbruik = min(verbruikTableCopy.loc[row_index, col_name], energy_per_quarter)
            if (battery_capacity_remaining >= verbruik):
                verbruikTableCopy.loc[row_index, col_name] -= verbruik;
                battery_capacity_remaining -= verbruik
            else:
                verbruikTableCopy.loc[row_index, col_name] -= battery_capacity_remaining
                battery_capacity_remaining = 0
                break   
        
    # Calculate total, average, min, and max for each row
    numeric_columns = [col for col in verbruikTableCopy.columns if col.lower() not in excluded_columns]
    verbruikTableCopy["total"] = verbruikTableCopy[numeric_columns].sum(axis=1)
    verbruikTableCopy["average"] = verbruikTableCopy[numeric_columns].mean(axis=1)
    verbruikTableCopy["minimum"] = verbruikTableCopy[numeric_columns].min(axis=1)
    verbruikTableCopy["maximum"] = verbruikTableCopy[numeric_columns].max(axis=1)
    return verbruikTableCopy

geoptimaliseerdVerbruikTable = process_charging_tables(
    verbruikTable, 
    create_charge_table(prijzenTable),
    create_discharge_table(prijzenTable),
    LAAD_VERMOGEN,
    BATTERIJ_CAPACITEIT,
    RTE    
  );
# print(geoptimaliseerdVerbruikTable.head())
geoptimaliseerdVerbruikTable.to_csv("verbruik_optimaliseerd.csv", index=False)

resultTable = multiply_tables_by_column(geoptimaliseerdVerbruikTable, prijzenTable)

total_dyn_costs2 = sum_dataframe_values(resultTable)

print("Energiekosten met dynamisch contract en batterij: €", f"{total_dyn_costs2:.2f}, voordeel = €", f"{totalCost - total_dyn_costs2:.2f}")


Battery capacity: 2 kWh, charge speed: 2.5 kW
Number of charge intervals [15 min.]: 3.2, energy per interval: 0.625 kWh
Energiekosten met dynamisch contract en batterij: € 1060.41, voordeel = € 179.27


In [401]:

print("Energie gebruik:", energieUsage, "[kWh]")
print("Energie prijs:", energy_price, "[€/kWh]")
print("Energiekosten met vast contract: €", f"{totalCost:.2f}")
print("Energiekosten met dynamisch contract: €", f"{total_dyn_costs:.2f}, voordeel = €", f"{totalCost - total_dyn_costs:.2f}")
print("Energiekosten met dynamisch contract en batterij: €", f"{total_dyn_costs2:.2f}, voordeel = €", f"{totalCost - total_dyn_costs2:.2f}")
print("Batterij capaciteit:", BATTERIJ_CAPACITEIT, "[kWh]")
print("Batterij laadvermogen:", LAAD_VERMOGEN, "[kW]")
print("Batterij prijs:", BATTERIJ_PRIJS, "[€]")
print("Batterijvoordeel: €", f"{total_dyn_costs - total_dyn_costs2:.2f}")
print("Batterij terugverdientijd:", BATTERIJ_PRIJS / (total_dyn_costs - total_dyn_costs2), "[jaar]")

Energie gebruik: 4591.392 [kWh]
Energie prijs: 0.27 [€/kWh]
Energiekosten met vast contract: € 1239.68
Energiekosten met dynamisch contract: € 1142.14, voordeel = € 97.54
Energiekosten met dynamisch contract en batterij: € 1060.41, voordeel = € 179.27
Batterij capaciteit: 2 [kWh]
Batterij laadvermogen: 2.5 [kW]
Batterij prijs: 1250 [€]
Batterijvoordeel: € 81.73
Batterij terugverdientijd: 15.294240207125261 [jaar]
